<a href="https://colab.research.google.com/github/alireza1420/stargazer-prediction-pipeline/blob/carolines-neural-net/Best_Model_May_16.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from xgboost import XGBRegressor
import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score
import joblib

class GitHubRepoPreprocessor:
    def __init__(self, reference_date=None):
        self.date = reference_date or datetime(2025, 5, 1)
        self.numeric_features = [
            'forks', 'open_issues', 'size', 'subscribers_count',
            'contributors_count', 'commits_count', 'readme_size',
            'project_age', 'days_since_update', 'days_since_push',
            'forks_per_day', 'issues_per_day', 'update_rate'
        ]
        self.categorical_features = ["language", "license"]
        self.column_transformer = None

    def transform(self, df):
        df["created_at"] = pd.to_datetime(df["created_at"]).dt.tz_localize(None)
        df["updated_at"] = pd.to_datetime(df["updated_at"]).dt.tz_localize(None)
        df["pushed_at"] = pd.to_datetime(df["pushed_at"]).dt.tz_localize(None)

        df["project_age"] = (self.date - df["created_at"]).dt.days
        df["days_since_update"] = (self.date - df["updated_at"]).dt.days
        df["days_since_push"] = (self.date - df["pushed_at"]).dt.days

        df["license"] = df["license"].fillna("None")
        df["language"] = df["language"].fillna("Unknown")

        df["forks_per_day"] = df["forks"] / (df["project_age"] + 1)
        df["issues_per_day"] = df["open_issues"] / (df["project_age"] + 1)
        df["update_rate"] = 1 / (1 + df["days_since_update"])

        df.replace([np.inf, -np.inf], np.nan, inplace=True)
        df.dropna(inplace=True)

        df["has_wiki"] = df["has_wiki"].astype(int)
        df["has_projects"] = df["has_projects"].astype(int)
        df["has_downloads"] = df["has_downloads"].astype(int)
        df["is_fork"] = df["is_fork"].astype(int)
        df["archived"] = df["archived"].astype(int)

        selected_features = [
            'forks', 'open_issues', 'size', 'has_wiki', 'has_projects', 'has_downloads',
            'is_fork', 'archived', 'language', 'license',
            'subscribers_count', 'contributors_count', 'commits_count', 'readme_size',
            'project_age', 'days_since_update', 'days_since_push',
            'forks_per_day', 'issues_per_day', 'update_rate'
        ]

        return df[selected_features], df["stars"]

    def get_preprocessor(self):
        self.column_transformer = ColumnTransformer(
            transformers=[
                ("num", StandardScaler(), self.numeric_features),
                ("cat", OneHotEncoder(handle_unknown="ignore"), self.categorical_features)
            ],
            remainder='passthrough'
        )
        return self.column_transformer


df = pd.read_csv('github_repo_features.csv')
preprocessor = GitHubRepoPreprocessor()
X, y = preprocessor.transform(df)


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

preprocessor_pipeline = preprocessor.get_preprocessor()
X_train_transformed = preprocessor_pipeline.fit_transform(X_train)
X_test_transformed = preprocessor_pipeline.transform(X_test)


#define the model
model = Sequential()
model.add(tf.keras.Input(shape=(X_train_transformed.shape[1],)))
model.add(Dense(64, activation='relu'))
model.add(Dense(64, activation='relu'))
model.add(Dense(1))

optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
model.compile(loss='mse', optimizer=optimizer, metrics=['mae'])

#fit the model to the dataset
model.fit(X_train_transformed, y_train, epochs=150, batch_size=10, validation_split=0.2)

model.save('neural_network_new_model.h5')

# For prediction and evaluation, also use the transformed test data
predictions = model.predict(X_test_transformed)

#evaluate the model
_, R2_score = model.evaluate(X_test_transformed, y_test)
print('R2 score:', r2_score(y_test, predictions))

Epoch 1/150
64/64 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - loss: 3990032384.0000 - mae: 49843.9766 - val_loss: 4650639872.0000 - val_mae: 53572.4883
Epoch 2/150
64/64 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 3936191744.0000 - mae: 49252.4922 - val_loss: 4643265024.0000 - val_mae: 53507.4297
Epoch 3/150
64/64 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 4945590784.0000 - mae: 51372.3164 - val_loss: 4616123904.0000 - val_mae: 53271.1758
Epoch 4/150
64/64 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 4228304384.0000 - mae: 49318.6562 - val_loss: 4551488512.0000 - val_mae: 52711.5742
Epoch 5/150
64/64 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 5138715648.0000 - mae: 53299.9062 - val_loss: 4438046208.0000 - val_mae: 51718.5234
Epoch 6/150
64/64 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 4953272320.0000 - mae: 52229.3164 - val_loss: 4269574144.0000 - val_mae: 50209.8320
Epoch 7/150
64/64 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 4074244352.0000 - mae: 47134.0820 - val_loss: 4043101440.0000 - val_mae: 48110.160

7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 397782624.0000 - mae: 13666.4199 
R2 score: 0.4739364981651306
